# Assignment · Add a regional sales lookup

The sales API can find the leading region for a month and compare two months across all regions. Add a third kind of report question: **“What were North's sales in March?”** Then test the API, prepare a follow-up using saved context, and sketch how the same app could run on AWS.

Work through the parts in order. Use the existing project and keep the sample sales rows as they are. You'll edit one copy of the API script and record your checks and short answers here.

| Part | What to finish |
|---|---|
| 1 | Start the API and confirm the existing report works |
| 2 | Add a tool for one region and one month |
| 3 | Check the answer, tool use, and error responses |
| 4 | Build a follow-up request from conversation state |
| 5 | Explain how the API would run on AWS |

## 1. Start with the working API

Install the [module packages](README.md#setup). In this folder, copy `sales_agent_api.py` to `sales_agent_assignment_api.py`. Start the copy with either Ollama or Groq using the [API setup guide](04_sales_agent_api.md). In the guide's launch command, use `sales_agent_assignment_api:app` where it says `sales_agent_api:app`.

For example, with Ollama running and the model pulled, the default launch command is

~~~bash
python -m uvicorn sales_agent_assignment_api:app --port 8000
~~~

Keep the server terminal open for the logs. If the server uses another port, change `API_URL` below. Run the next two cells before changing your copy of the API.

In [ ]:
import httpx

API_URL = "http://127.0.0.1:8000"


def ask_api(route, question):
    # json= sends the dictionary as a JSON request body.
    return httpx.post(f"{API_URL}{route}", json={"question": question}, timeout=40)


health = httpx.get(f"{API_URL}/health", timeout=5)
health.raise_for_status()
print(health.json())  # Expected: {'status': 'ok'}

In [ ]:
response = ask_api("/report", "Compare total sales in February and March.")
response.raise_for_status()
print(response.json()["answer"])

Expect February at $1,800 and March at $2,700. Find `compare_months` in the terminal log. In one sentence, explain why $2,700 is different from North's March figure of $1,200.

**Your answer** …

## 2. Add a tool for one region

In `sales_agent_assignment_api.py`, add the following function above `report_agent` and complete the body.

~~~python
@tool
def sales_for_region(month: str, region: str) -> dict:
    """Return sales for one region and one full month name."""
    # Reuse totals_by_region(month), then pick out the requested region.
    raise NotImplementedError
~~~

- Accept extra spaces and different capitalization. For example, `" march "` and `" north "` should resolve to March and North.
- Return a dictionary with `month`, `region`, and `sales`. Return `None` for `sales` when there is no matching row. Python's `None` becomes JSON `null` when serialized.
- Add the tool to the existing `report_agent` tool list. Add an instruction telling the model to use it for one region in one month and to explain when the tool returns no data. Keep both existing report tools available.
- Add an info log **inside the new tool** that shows its name and the two cleaned arguments. Use the existing `logger`.

Use the examples below to check your calculation before reviewing the wording of the model reply.

| Tool arguments | Expected tool result |
|---|---|
| `month=" march ", region=" north "` | `{"month": "March", "region": "North", "sales": 1200}` |
| `month="February", region="North"` | `{"month": "February", "region": "North", "sales": None}` |
| `month="April", region="North"` | `{"month": "April", "region": "North", "sales": None}` |

**Hint** `totals_by_region("March")` returns a dictionary with region names as keys. Call `.get()` with a cleaned region name to get its value or `None`.

Restart Uvicorn after editing the file. Check that `POST /report` now answers “What were North's sales in March?”

## 3. Check behavior, not just wording

Run the cases below. Keep `raise_for_status()` on the successful request in Part 1. Here, print each status and response body first so you can examine the expected `422` as well as the successful replies.

| Case | Expected status | What to check in the reply and log |
|---|---|---|
| North in March | `200` | North and $1,200. `sales_for_region` ran with March and North. |
| North in April | `200` | No data for North in April. The tool ran with April and North. |
| Compare February and March | `200` | $1,800 and $2,700. The report still uses `compare_months`. |
| Missing month on `/chat` | `200` | Asks which month to check. The handler returns before an agent run. |
| Spaces only on `/chat` | `422` | The body has `detail`. The handler rejects the input before an agent run. |

**Your task** Complete the checks beneath the loop. Check the status and response content for each case. Allow wording changes in model replies. Look at the terminal for the three report cases and paste the relevant tool names and arguments in the notes below the cell.

In [ ]:
cases = {
    "regional_total": ("/report", "What were North's sales in March?"),
    "no_data": ("/report", "What were North's sales in April?"),
    "comparison": ("/report", "Compare total sales in February and March."),
    "missing_month": ("/chat", "Which region led?"),
    "blank": ("/chat", "   "),
}

responses = {}
for name, (route, question) in cases.items():
    response = ask_api(route, question)
    responses[name] = response
    print(name, response.status_code, response.json())

# Example: a blank body value is rejected before calling the agent.
assert responses["blank"].status_code == 422
assert responses["blank"].json()["detail"] == "Please enter a question."

# TODO: Add assertions for the four other cases.
# Use .lower() for region and month names, and remove commas before checking amounts.

**Tool log notes** Record which tools ran and the arguments logged by your new tool. Explain why “North, $1,200” could still be a failed run if the tool was called with the wrong month.

**Reliability questions**

1. Look at `run_agent()`. What is the difference between `await agent_call` and `await asyncio.wait_for(agent_call, timeout=30)`? What does this API return if the run exceeds its wait budget?
2. What is the difference between receiving an HTTP error response and failing to connect to the server at all? At which line would `raise_for_status()` be useful?
3. The sales lookup only reads data. If the lookup times out, what could one bounded retry do? If the tool changed a sales record instead, what would you check before repeating that action?

Write your answers here.

## 4. Use context for a follow-up

The next question is “Write the same briefing for this region.” That message leaves out both the region and month. Build the next model input in the notebook using the saved conversation fields, the saved format, and the result you just fetched.

**Your task** Fill in `briefing_context` with `task`, `period`, `region`, `format`, and `sales_facts`. Use the user's ID when loading the format and both IDs when loading the conversation. Put the answer from `regional_total` in `sales_facts` as a one-item list.

Then make a copy for a one-time override to “One short paragraph.” Print both formats to show that the saved preference still says “Two short bullets.” This exercise assembles the input in the notebook. A future briefing route could send it to a model.

In [ ]:
from pprint import pprint

user_id = "priya"
conversation_id = "march-report"

conversations = {
    ("priya", "march-report"): {"month": "March", "region": "North"},
}
preferences = {"priya": {"format": "Two short bullets"}}

# TODO: Load the fields and the successful regional_total response.
briefing_context = {"task": "Write a short sales briefing."}
pprint(briefing_context, sort_dicts=False)

# TODO: Make a separate context for the one-time format override.
# Check that preferences[user_id]["format"] still has its original value.

**Check your context** It should have March, North, the two-bullet preference, and the $1,200 evidence. If the chat grows to 100 turns, explain in two sentences what you would include in the next model call and when you would look up an older message.

**Your answer** …

## 5. Sketch a deployment

Use the [AWS overview](07_deploying_agentic_systems_on_aws.md) and draw the path of one request. A text diagram is enough. Include the browser, load balancer, FastAPI backend, model, and Python tool. Label which part handles the Groq key if you use Groq.

Answer these questions below your sketch.

1. Which service could run the FastAPI container? Where could the sales file live?
2. Where does the model run with a hosted API? Where could an open-weight model run if you serve it yourself?
3. Which route could a load balancer use for a health check? Which request would you send separately to see if the model and tool work?

**Your sketch and answers** …

## Before submitting

- Include this notebook with your assertions, tool log notes, context, and deployment sketch.
- Include `sales_agent_assignment_api.py` with the new tool and updated report agent.
- Run the API and notebook from the top once. Keep the output small and leave local credentials, such as `.env`, out of your submission.

If you get stuck, refer to [the API walkthrough](00_sales_agent_api_and_evals.ipynb), [the tool eval lab](02_sales_agent_design_and_evals.ipynb), or [the context and memory lab](05_context_memory_and_agent_harness.ipynb).